<a href="https://colab.research.google.com/github/ritakimani9-lang/machinelearning/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ritakimani9-lang/machinelearning/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Content lifecycle: growing vs. declining pages

The paper compares growing and declining pages and reports that growing pages were younger on average: 185 days compared with 228 days for declining pages. Word count was nearly identical between the groups (1,487 vs. 1,481 words). The paper presents age as the clearest observed difference between the two groups.

**My methodology question:** How exactly was the growing-versus-declining label constructed, including the time window used to define the outcome? I would also want to confirm that the features being compared were measured strictly before the outcome window. This would help rule out the possibility that information from the label period is included in the explanatory variables.

### Finding 2 — Growth prediction on familiar vs. unseen brands

The paper reports that its growth model achieved about 90% accuracy on new pages from brands represented in training, compared with 75% accuracy on brands that were not seen during training. The reported range for the unseen-brand evaluation was 64%–85%.

**My methodology question:** Does the validation design match the generalization claim? In particular, I would want to understand how the brand-level split was constructed and confirm that pages from a brand held out for evaluation could not contribute information to training. This distinction matters because performance on new pages from familiar brands is a different question from performance on a completely unseen brand.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Before vs. after: validation split

In Week 5, I evaluated the model using a client-grouped split because multiple content items can belong to the same client. This prevents content from the same client appearing in both training and evaluation folds.

For comparison, I will reproduce the earlier random-split evaluation and then evaluate the same model using client-grouped cross-validation. The purpose is to measure how much the evaluation result changes when the split better matches the intended use case of evaluating on an unseen client.

I will report the model metric alongside the test base rate so that the metric is interpreted relative to the class distribution.


In [2]:
# Section 2 — Load data and recreate the Week-5 feature set

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

SEED = 42

# Load the same dataset used in Week 5

URL = 'https://raw.githubusercontent.com/ritakimani9-lang/machinelearning/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(URL)

# Recreate the Week-5 label
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

# Exact Week-5 feature set
NUM = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "engaged_sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "days_with_impressions",
    "days_with_sessions",
]

CAT = [
    "content_type",
    "main_intent",
    "competition_level",
]

LEAKY = [
    "trend_pct",
    "trend_direction",
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "sessions_last_30d",
    "sessions_prev_30d",
    "is_declining_label",
    "content_id",
    "client_id",
]

# Safety check: no feature may be in the leakage list
assert not set(NUM + CAT) & set(LEAKY), "Leaky column in features!"

X = df[NUM + CAT].copy()
y = df["is_declining_label"].copy()
groups = df["client_id"].copy()

print("Rows:", len(df))
print("Model features:", len(NUM + CAT))
print("Numeric features:", len(NUM))
print("Categorical features:", len(CAT))
print("Target base rate:", round(y.mean(), 3))
print("Number of clients:", groups.nunique())

print("\nExcluded leakage columns:")
print(LEAKY)

Rows: 30000
Model features: 20
Numeric features: 17
Categorical features: 3
Target base rate: 0.542
Number of clients: 32

Excluded leakage columns:
['trend_pct', 'trend_direction', 'impressions_last_30d', 'impressions_prev_30d', 'clicks_last_30d', 'clicks_prev_30d', 'sessions_last_30d', 'sessions_prev_30d', 'is_declining_label', 'content_id', 'client_id']


In [3]:
# Section 2 — Before vs. after: random split and client-grouped split

import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

SEED = 42

# ---------------------------------------------------------
# 1. Load the same dataset used in Week 5
# ---------------------------------------------------------

URL = "https://raw.githubusercontent.com/ritakimani9-lang/machinelearning/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(URL)

# Recreate the Week-5 label
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

# ---------------------------------------------------------
# 2. Exact Week-5 feature set
# ---------------------------------------------------------

NUM = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "engaged_sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "days_with_impressions",
    "days_with_sessions",
]

CAT = [
    "content_type",
    "main_intent",
    "competition_level",
]

LEAKY = [
    "trend_pct",
    "trend_direction",
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "sessions_last_30d",
    "sessions_prev_30d",
    "is_declining_label",
    "content_id",
    "client_id",
]

assert not set(NUM + CAT) & set(LEAKY), "Leaky column found in feature set."

X = df[NUM + CAT].copy()
y = df["is_declining_label"].copy()
groups = df["client_id"].copy()

print("DATA CHECK")
print("Rows:", len(df))
print("Model features:", len(NUM + CAT))
print("Numeric features:", len(NUM))
print("Categorical features:", len(CAT))
print("Clients:", groups.nunique())
print("Overall base rate:", round(y.mean(), 3))


# ---------------------------------------------------------
# 3. Preprocessing + Week-5 Random Forest
# ---------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            make_pipeline(
                SimpleImputer(strategy="median"),
                StandardScaler()
            ),
            NUM
        ),
        (
            "cat",
            make_pipeline(
                SimpleImputer(strategy="most_frequent"),
                OneHotEncoder(handle_unknown="ignore")
            ),
            CAT
        )
    ]
)

rf = RandomForestClassifier(
    300,
    min_samples_leaf=20,
    n_jobs=-1,
    random_state=SEED
)

def make_model():
    return make_pipeline(
        preprocessor,
        rf
    )


def precision_at_k(scores, labels, k=100):
    order = np.argsort(-scores, kind="stable")[:k]
    return labels.iloc[order].mean()


# ---------------------------------------------------------
# 4. BEFORE — random stratified split
# ---------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

random_model = make_model()
random_model.fit(X_train, y_train)

random_prob = random_model.predict_proba(X_test)[:, 1]

random_auc = roc_auc_score(y_test, random_prob)
random_p100 = precision_at_k(random_prob, y_test, 100)
random_base_rate = y_test.mean()

print("\nBEFORE — Random stratified split")
print("--------------------------------")
print(f"Test rows:       {len(y_test):,}")
print(f"Test base rate:  {random_base_rate:.3f}")
print(f"ROC-AUC:         {random_auc:.3f}")
print(f"Precision@100:   {random_p100:.3f}")


# ---------------------------------------------------------
# 5. AFTER — client-grouped 5-fold split
# ---------------------------------------------------------

group_cv = GroupKFold(n_splits=5)

group_rows = []

for fold, (train_idx, test_idx) in enumerate(
    group_cv.split(X, y, groups)
):
    model = make_model()

    model.fit(
        X.iloc[train_idx],
        y.iloc[train_idx]
    )

    prob = model.predict_proba(
        X.iloc[test_idx]
    )[:, 1]

    fold_auc = roc_auc_score(
        y.iloc[test_idx],
        prob
    )

    fold_p100 = precision_at_k(
        prob,
        y.iloc[test_idx],
        100
    )

    group_rows.append({
        "fold": fold,
        "base_rate": y.iloc[test_idx].mean(),
        "test_clients": groups.iloc[test_idx].nunique(),
        "roc_auc": fold_auc,
        "precision_at_100": fold_p100
    })

group_results = pd.DataFrame(group_rows)

print("\nAFTER — Client-grouped 5-fold split")
print("-----------------------------------")
print(group_results.round(3).to_string(index=False))

print("\nGrouped mean ± std")
print("------------------")

group_summary = group_results[
    ["base_rate", "roc_auc", "precision_at_100"]
].agg(["mean", "std"]).T

print(group_summary.round(3))


# ---------------------------------------------------------
# 6. Before vs. after comparison
# ---------------------------------------------------------

group_auc_mean = group_results["roc_auc"].mean()
group_p100_mean = group_results["precision_at_100"].mean()
group_base_mean = group_results["base_rate"].mean()

comparison = pd.DataFrame({
    "evaluation": [
        "Random stratified split",
        "Client-grouped 5-fold"
    ],
    "base_rate": [
        random_base_rate,
        group_base_mean
    ],
    "roc_auc": [
        random_auc,
        group_auc_mean
    ],
    "precision_at_100": [
        random_p100,
        group_p100_mean
    ]
})

print("\nBEFORE vs AFTER")
print("---------------")
print(comparison.round(3).to_string(index=False))


# ---------------------------------------------------------
# 7. Validation gap
# ---------------------------------------------------------

print("\nObserved validation gap")
print("------------------------")
print(
    "Precision@100 difference:",
    round(random_p100 - group_p100_mean, 3)
)

print(
    "ROC-AUC difference:",
    round(random_auc - group_auc_mean, 3)
)


# ---------------------------------------------------------
# 8. Error examples from one grouped test fold
# ---------------------------------------------------------

# Use fold 0 for concrete error examples.
train_idx, test_idx = list(
    group_cv.split(X, y, groups)
)[0]

error_model = make_model()
error_model.fit(
    X.iloc[train_idx],
    y.iloc[train_idx]
)

error_test = df.iloc[test_idx].copy()

error_test["predicted_probability"] = (
    error_model.predict_proba(
        X.iloc[test_idx]
    )[:, 1]
)

# False positives = model ranked highly but label is 0
false_positives = (
    error_test[
        error_test["is_declining_label"] == 0
    ]
    .sort_values(
        "predicted_probability",
        ascending=False
    )
    .head(5)
)

print("\nFALSE-POSITIVE EXAMPLES")
print("-----------------------")

print(
    false_positives[
        [
            "predicted_probability",
            "avg_position",
            "content_age_days",
            "ctr",
            "impressions_90d",
            "days_since_last_update",
            "trend_direction",
            "trend_pct"
        ]
    ].round(3).to_string(index=False)
)

print("\nNote: client_id and content_id are intentionally not displayed.")


# ---------------------------------------------------------
# 9. Simple feature sanity check
# ---------------------------------------------------------

print("\nFeature sanity check")
print("--------------------")

print(
    "Top model inputs from the Week-5 analysis included "
    "avg_position and content_age_days."
)

print(
    "These are available in the final feature set and are not "
    "direct components of the declining label."
)

DATA CHECK
Rows: 30000
Model features: 20
Numeric features: 17
Categorical features: 3
Clients: 32
Overall base rate: 0.542

BEFORE — Random stratified split
--------------------------------
Test rows:       6,000
Test base rate:  0.542
ROC-AUC:         0.767
Precision@100:   0.930

AFTER — Client-grouped 5-fold split
-----------------------------------
 fold  base_rate  test_clients  roc_auc  precision_at_100
    0      0.490             1    0.648              0.84
    1      0.645             7    0.632              0.86
    2      0.379             8    0.744              0.85
    3      0.622             8    0.683              0.68
    4      0.585             8    0.688              0.78

Grouped mean ± std
------------------
                   mean    std
base_rate         0.544  0.110
roc_auc           0.679  0.043
precision_at_100  0.802  0.075

BEFORE vs AFTER
---------------
             evaluation  base_rate  roc_auc  precision_at_100
Random stratified split      0.542    

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit

I checked the final feature set against three main leakage risks.

First, I checked for label-derived and sibling columns. The declining label is derived from the change in impressions between the last 30 days and the previous 30 days, so `trend_pct`, `trend_direction`, and the last/previous-30-day metrics must not be model features. The content and client IDs are also excluded from the feature set.

Second, I checked for decision-derived features. No existing product score or decision flag is included in the final model features.

Third, I checked the time windows. The final feature set contains 90-day metrics such as `impressions_90d`, `clicks_90d`, and `sessions_90d`. These windows overlap the period used to construct the declining label, so they remain a limitation. A client-grouped split does not fix same-window overlap. I therefore treat the current model as directional decision-support rather than evidence of a causal or deployment-ready prediction system.

The direct label leakage checks pass, but the same-window feature issue remains an important limitation.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3 — Leakage audit

# ---------------------------------------------------------
# 1. Check direct label-derived columns
# ---------------------------------------------------------

direct_leakage = [
    "trend_pct",
    "trend_direction",
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "sessions_last_30d",
    "sessions_prev_30d"
]

used_features = set(NUM + CAT)

direct_leaks_in_features = [
    col for col in direct_leakage
    if col in used_features
]

print("Direct/sibling leakage columns in features:")
print(direct_leaks_in_features)

assert len(direct_leaks_in_features) == 0


# ---------------------------------------------------------
# 2. Check IDs
# ---------------------------------------------------------

ids = ["client_id", "content_id"]

ids_in_features = [
    col for col in ids
    if col in used_features
]

print("\nIDs in model features:")
print(ids_in_features)

assert len(ids_in_features) == 0


# ---------------------------------------------------------
# 3. Check decision-derived/product flags
# ---------------------------------------------------------

decision_like_columns = [
    col for col in df.columns
    if any(
        word in col.lower()
        for word in [
            "score",
            "flag",
            "decision",
            "recommendation"
        ]
    )
]

decision_features_used = [
    col for col in decision_like_columns
    if col in used_features
]

print("\nDecision-derived columns used as features:")
print(decision_features_used)


# ---------------------------------------------------------
# 4. Identify same-window features
# ---------------------------------------------------------

same_window_features = [
    col for col in [
        "impressions_90d",
        "clicks_90d",
        "pageviews_90d",
        "sessions_90d",
        "users_90d",
        "engaged_sessions_90d",
        "ai_sessions_90d",
        "scroll_events_90d",
        "ctr",
        "engagement_rate",
        "scroll_rate",
        "ai_traffic_pct"
    ]
    if col in used_features
]

print("\nPotential same-window features:")
print(same_window_features)


# ---------------------------------------------------------
# 5. Final audit result
# ---------------------------------------------------------

print("\nLEAKAGE AUDIT RESULT")
print("--------------------")

if (
    len(direct_leaks_in_features) == 0
    and len(ids_in_features) == 0
    and len(decision_features_used) == 0
):
    print("Direct label/sibling leakage: PASS")
    print("ID leakage: PASS")
    print("Decision-derived feature check: PASS")
else:
    print("One or more direct leakage checks require review.")

print(
    "\nLimitation: the 90-day features overlap the label period "
    "and therefore require a future time-aligned feature design "
    "before stronger deployment claims."
)

Direct/sibling leakage columns in features:
[]

IDs in model features:
[]

Decision-derived columns used as features:
[]

Potential same-window features:
['impressions_90d', 'clicks_90d', 'sessions_90d', 'engaged_sessions_90d', 'ctr', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

LEAKAGE AUDIT RESULT
--------------------
Direct label/sibling leakage: PASS
ID leakage: PASS
Decision-derived feature check: PASS

Limitation: the 90-day features overlap the label period and therefore require a future time-aligned feature design before stronger deployment claims.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite

**Earlier claim:**
“The Random Forest model can identify declining content and should be used to prioritize pages for review.”

**Why this goes too far:**
The grouped evaluation is based on only 32 clients, the fold base rates vary substantially, and the model contains 90-day features that overlap the label period. The validation therefore does not support a strong claim that the model will generalize to future production data.

**Rewritten claim:**
“Under client-grouped cross-validation, the Random Forest showed measured directional signal for ranking pages associated with the declining label, with Precision@100 and ROC-AUC reported alongside the test base rate. The result can be used as decision-support evidence for further investigation, but the current evaluation does not establish future-period performance or deployment readiness because some feature windows overlap the label period.”


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 — Claim rewrite backed by the measured results

print("CLAIM EVIDENCE")
print("--------------")

print(
    f"Random-split Precision@100: {random_p100:.3f}"
)

print(
    f"Grouped Precision@100: {group_p100_mean:.3f}"
)

print(
    f"Random-split ROC-AUC: {random_auc:.3f}"
)

print(
    f"Grouped ROC-AUC: {group_auc_mean:.3f}"
)

print(
    f"Grouped mean test base rate: {group_base_mean:.3f}"
)

print(
    "\nInterpretation: the measured result changes with the "
    "validation design. The grouped result is therefore used "
    "as the more relevant directional estimate for unseen-client "
    "evaluation."
)

CLAIM EVIDENCE
--------------
Random-split Precision@100: 0.930
Grouped Precision@100: 0.802
Random-split ROC-AUC: 0.767
Grouped ROC-AUC: 0.679
Grouped mean test base rate: 0.544

Interpretation: the measured result changes with the validation design. The grouped result is therefore used as the more relevant directional estimate for unseen-client evaluation.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.